# 📊 Análise de Vendas e Impacto de Descontos
## 1. Contexto de negócio
Em mercados competitivos, descontos são frequentemente utilizados como mecanismo para estimular vendas, aumentar participação de mercado e acelerar o giro comercial.
Entretanto, existe uma questão estratégica relevante:
até que ponto conceder descontos contribui para o crescimento do negócio sem comprometer sua rentabilidade?
Este projeto investiga essa relação a partir de uma base transacional de vendas, buscando entender como diferentes níveis de desconto impactam faturamento, lucro, margem e ticket médio.
## 2. Objetivo da análise
O objetivo central deste estudo é avaliar se a política comercial de descontos está efetivamente gerando valor econômico para a operação.
Mais especificamente, a análise busca responder:
* Os descontos aumentam resultado financeiro ou apenas reduzem receita?
* Existe um limite a partir do qual o desconto passa a destruir margem?
* Quais regiões e categorias são mais sensíveis à política de preços?
* O comportamento se mantém consistente ao longo do tempo?
## 3. Fonte dos dados
A base utilizada neste estudo é o conjunto sales dataset, disponibilizado por VINOTH KANNA na plataforma Kaggle.
Características da base:
* 1.000 registros de vendas
* Período entre janeiro de 2023 e janeiro de 2024
* Comparação entre cenários com e sem desconto;
* Variáveis de preço, custo, desconto, categoria, região, canal e cliente
* Endereço: https://www.kaggle.com/datasets/vinothkannaece/sales-dataset

---

## 4. Bibliotecas utilizadas

In [ ]:
# Bibliotecas utilizadas
import pandas as pd
import import_ipynb
from pathlib import Path
from pyprojroot import here

---

## 5. Imports

In [ ]:
from sales_analysis.utils.csv_read.csv_load import *
from sales_analysis.utils.quality.quality_test import *
from sales_analysis.tables.table_kpis_list import *
from sales_analysis.dashboard.dashboard import *
from sales_analysis.graphics.graphic_months import *
from sales_analysis.graphics.graphic_discount_margin import *
from sales_analysis.graphics.graphic_region_category import *
from sales_analysis.graphics.graphic_impact_discount import *
from sales_analysis.tables.table_progressive_discounts import *

## 6. Leitura dos dados

In [ ]:
ROOT = here()
file = Path(ROOT / "data/sales_data.csv")

In [ ]:
df = load_csv(file)

---

## 7. Visualização inicial dos dados

In [ ]:
df.head(5)

### Observações iniciais
* O dataset contém informações sobre vendas
* Há variáveis categóricas(ex: Region), numéricas(ex: Sales_Amount) e do tipo data(ex: Sale_Date)
* Aparentemente sem valores nulos, incorretos, duplicados ou inconsistentes.

---

## 8. Estrutura dos dados
Nesta etapa, analisamos os tipos de dados, a quantidade de registros, a quantidade e nomes das colunas e memória utilizada.

In [ ]:
df.info()

### Principais pontos
* O dataset possui 1000 linhas e 14 colunas
* Variáveis numéricas estão representadas como `int64` e `float64`
* Variáveis categóricas estão como `object` (texto)
* A coluna `Sale_Date` precisará de conversão para `datetime`
* Sem valores nulos

---

## 9. Teste de qualidade dos dados
A função implementa uma série de testes para garantir a qualidade dos dados

In [ ]:
get_quality_test(df)

### Resultado da validação
* A base apresentou boa consistência estrutural.
* No entanto, além da data com tipo errado, surgiu um primeiro sinal relevante:
* 60,4% das transações com desconto apresentaram preço líquido inferior ao custo unitário.
* Esse achado já sugere que a política comercial pode estar comprometendo a rentabilidade operacional.

---

## ⚠️ Nota metodológica sobre Faturamento e Sales_Amount
De acordo com o dicionário do conjunto de dados, Sales_Amount representa o valor total da venda, já considerando eventuais descontos aplicados.

Neste projeto, optou-se por não utilizar Sales_Amount como base principal das métricas de faturamento, lucro e margem. Em vez disso, foi adotada uma métrica analítica construída a partir de Quantity_Sold, Unit_Price, Unit_Cost e Discount.

Essa escolha metodológica permite analisar de forma mais direta o efeito operacional de preço, volume e desconto sobre a rentabilidade, preservando consistência interna nas métricas utilizadas ao longo da análise.

---

## 10. Conversão de tipo
A coluna data de venda (`Sale_Date`) será convertida para o tipo `datatime`

In [ ]:
df["Sale_Date"] = pd.to_datetime(df["Sale_Date"], errors="coerce")

---

## 11. Criação dos principais KPIs

In [ ]:
# KPIs Financeiros - Métricas base
df["faturamento_sem_desc"] = df["Quantity_Sold"] * df["Unit_Price"]
df["faturamento_com_desc"] = df["Quantity_Sold"] * df["Unit_Price"] * (1 - df["Discount"])
df["custo"] = df["Quantity_Sold"] * df["Unit_Cost"]
df["lucro_sem_desc"] = df["faturamento_sem_desc"] - df["custo"]
df["lucro_com_desc"] = df["faturamento_com_desc"] - df["custo"]
df["margem_com_desc"] = df["lucro_com_desc"] / df["faturamento_com_desc"]
df["margem_sem_desc"] = df["lucro_sem_desc"] / df["faturamento_sem_desc"]

---

## 12. Estatística

In [ ]:
df.describe(include="number")

### Interpretação
A base analisada contém 1.000 registros de vendas, o que fornece uma amostra consistente para avaliação do desempenho comercial.

### Visão geral das vendas
O volume médio por transação é de 25,4 unidades (Quantity_Sold), com grande dispersão (std = 14,16), o que indica heterogeneidade relevante entre pedidos, coexistem vendas pequenas e grandes.

Em média, cada transação apresentou quantidade vendida de 25 unidades, com preço unitário médio de 2.728,44 e custo unitário médio de 2.475,30. O desconto médio aplicado foi de 15,24%, indicando uma política comercial relativamente frequente de concessão de descontos.

Em termos financeiros, o faturamento médio sem desconto foi de 70.329,94, enquanto o faturamento médio com desconto caiu para 59.686,17, evidenciando impacto direto das reduções de preço sobre a receita. O custo médio por operação foi de 63.842,09.

### Ao comparar os cenários, com e sem descontos, observa-se que:
* Em 50% das vendas, o lucro com desconto ficou abaixo de -1.210,46.
* Já sem desconto, a mediana do lucro foi 5.236,83.
Isso reforça que o impacto negativo dos descontos não se limita a poucos casos extremos, mas aparece de forma recorrente em boa parte das operações.

Conclusão: de forma geral, os dados indicam que a empresa possui estrutura de preço capaz de gerar lucro sem descontos, porém o nível médio de desconto atualmente praticado compromete a margem operacional, sugerindo a necessidade de revisão da política comercial ou segmentação mais criteriosa dos descontos concedidos.

---

## 13. DashBoard do comportamento comercial
⚠️ Observação: as métricas utilizadas neste dashboard são todas sem desconto. O motivo desta escolha foi demonstrar o comportamento 
comercial independente dos descontos.

In [ ]:
get_dashboard(df)

---

## 14. Diagnóstico
### 14.1 Gráfico de evolução mensal

In [ ]:
get_graphic_months(df)

### Interpretação da Evolução Mensal
No cenário com desconto, observa-se que, embora o faturamento se mantenha em patamares relevantes ao longo dos meses, a redução da receita líquida pressiona o resultado operacional, fazendo com que o lucro apresente maior volatilidade e, em determinados períodos, fique mais próximo do ponto de equilíbrio ou até negativo.

No cenário sem desconto, a trajetória do faturamento permanece superior, refletindo diretamente em níveis de lucro mais consistentes ao longo do tempo. Como o custo tende a acompanhar o volume de vendas, a principal diferença entre os dois cenários está no efeito dos descontos sobre a margem final.

De forma geral, a análise mensal indica que a política de descontos reduz a capacidade de geração de resultado, ainda que o comportamento sazonal das vendas permaneça semelhante entre os dois cenários.

---

### 14.2 Gráfico de Desconto vs Margem

In [ ]:
get_graphic_discount_margin(df)

### Interpretação de Desconto vs Margem
O gráfico de dispersão mostra a relação entre o percentual de desconto e a margem de lucro das vendas.

No cenário com desconto, observa-se uma tendência de redução da margem à medida que o desconto aumenta. Embora exista dispersão entre os pontos, a concentração de valores em faixas de margem mais baixas — inclusive negativas — indica que descontos mais elevados tendem a pressionar diretamente a rentabilidade das operações.

No cenário sem desconto, a margem permanece majoritariamente positiva e mais estável, sem apresentar deterioração acentuada ao longo dos diferentes níveis de desconto. Isso ocorre porque, nesse caso, a margem é calculada com base no preço cheio, preservando maior capacidade de geração de lucro.

De forma geral, o gráfico sugere que o aumento dos descontos está associado à compressão das margens, reforçando que a política comercial impacta diretamente o resultado financeiro.

---

### 14.3 Gráfico de Análise comparativa do lucro por região e categoria

In [ ]:
get_graphic_region_category(df)

### Interpretação da Análise comparativa de lucro por região e categoria
O gráfico de barras compara o desempenho do lucro com desconto e sem desconto, permitindo avaliar o impacto da política comercial em diferentes segmentos do negócio.

Na análise por região, observa-se que o cenário sem desconto apresenta resultados superiores em praticamente todas as localidades, indicando que os descontos reduzem de forma consistente a rentabilidade regional. A diferença entre as barras também sugere que algumas regiões são mais sensíveis à política de descontos do que outras, apresentando maior compressão de margem.

Na análise por categoria de produto, o comportamento é semelhante. Em geral, o lucro sem desconto permanece acima do lucro com desconto, evidenciando que o impacto dos descontos não ocorre de forma homogênea entre as categorias. Algumas linhas de produto conseguem preservar parte da rentabilidade, enquanto outras apresentam redução mais acentuada no resultado.

De forma geral, o gráfico indica que, embora os descontos possam contribuir para a dinâmica comercial, seu efeito financeiro tende a reduzir o lucro em diferentes regiões e categorias, reforçando a importância de avaliar a concessão de descontos de forma segmentada.

---

### 14.4 Gráfico de impacto do desconto sobre o lucro

In [ ]:
get_graphic_impact_discount(df)

### Interpretação do impacto do desconto sobre o lucro
O gráfico de simulação mostra a relação entre o percentual de desconto aplicado e o lucro total estimado da operação, assumindo que o volume vendido e a estrutura de custos permanecem constantes.

Observa-se uma trajetória claramente descendente: à medida que o desconto aumenta, o lucro se reduz de forma progressiva.

Esse comportamento ocorre porque a redução do preço afeta diretamente o faturamento, enquanto o custo total permanece praticamente inalterado. Como consequência, cada incremento no nível de desconto comprime a margem operacional e reduz a capacidade de geração de resultado.

O ponto mais relevante do gráfico é a interseção da reta com a linha de lucro zero.

Esse ponto representa o limite econômico de equilíbrio da política comercial.

---

## 15. Comparação de Métricas com e sem Descontos

In [ ]:
get_list_kpis(df)

### Interpretação das métricas consolidadas
A comparação entre os cenários com e sem desconto mostra um impacto financeiro bastante relevante da política comercial sobre o desempenho da operação.

Sem descontos, o faturamento total alcançaria 70,33 milhões. Com a aplicação dos descontos observados na base, esse valor cai para 59,69 milhões, uma redução de aproximadamente 10,64 milhões, ou cerca de 15,1% da receita potencial.

Embora o custo total permaneça em 63,84 milhões, a redução de receita altera completamente o resultado operacional.

No cenário sem desconto, a operação geraria 6,49 milhões de lucro, com margem positiva de 9,22%. Já no cenário com desconto, o resultado se torna negativo em 4,16 milhões, levando a margem para -6,96%.

---

## 17. Efeito progressivo dos descontos sobre o lucro

In [ ]:
get_graphic_progressive_discounts(df)

### A descoberta principal — existe um ponto crítico
Foi realizada uma simulação para avaliar o efeito progressivo dos descontos sobre o lucro.

O resultado mostrou um ponto de ruptura bastante claro.

#### Faixa de equilíbrio:
até aproximadamente 9% de desconto, o negócio ainda permanece próximo do equilíbrio

#### Ponto crítico:
a partir de 9%, o lucro passa a ser negativo

---

## 18. Conclusão executiva

A análise mostra que:

#### A operação é economicamente viável sem descontos:
A estrutura de custos e preços permite geração positiva de lucro.

#### A política atual de descontos compromete a rentabilidade:
O desconto médio observado (15,24%) já se encontra acima da faixa economicamente saudável.

#### O principal problema não é vender pouco:
O principal problema é vender com margem insuficiente.

---

## 19. Recomendações de negócio

Com base nos achados, três recomendações práticas se destacam.

#### Revisar o teto de desconto:
O desconto acima de 9% deve ser tratado como zona de atenção.

#### Segmentar a política comercial
Descontos devem ser diferenciados por:

* região
* categoria
* perfil de cliente
* canal de venda

#### Tratar desconto como alavanca tática:
Desconto deve ser ferramenta de estímulo comercial, e não prática recorrente que comprometa margem.

#### Política sugerida:
até 5% → desconto liberado
5% a 9% → permitido com justificativa comercial
acima de 9% → apenas campanhas específicas ou clientes estratégicos

---

## 20. Encerramento

Este estudo mostra que aumentar vendas nem sempre significa aumentar resultado.

A empresa continua vendendo.

Mas, ao conceder descontos em níveis excessivos, parte relevante dessas vendas deixa de gerar valor econômico.

#### O verdadeiro desafio não é vender mais. É vender preservando rentabilidade.

---

## Sobre o autor

* Autor: Carlos da Costa
* Recife, PE - Brasil
* Telefone: +55 81 99712 9140
* Telegram: @jcarlossc
* Blogger linguagem R: https://informaticus77-r.blogspot.com/
* Blogger linguagem Python: https://informaticus77-python.blogspot.com/
* Email: jcarlossc1977@gmail.com
* LinkedIn: https://www.linkedin.com/in/carlos-da-costa-669252149/
* GitHub: https://github.com/jcarlossc
* Kaggle: https://www.kaggle.com/jcarlossc/
* Twitter/X: https://x.com/jcarlossc1977

---